# 01_train.ipynb — Heston DDN 模型训练（IV 目标 + 乘法 Trick 梯度正则化）

本 Notebook 负责：
1. 加载（或生成）训练数据 — 目标为 **BS 隐含波动率 (IV)**
2. 训练数据已预过滤满足 **Feller 条件** ($2\kappa\theta > \sigma^2$)
3. 按 70:15:15 划分数据集
4. 训练 HestonDDN（预测 IV，最后一层 Softplus 保证非负）
5. **乘法 Trick 梯度损失**（解决 Vega→0 时除以零爆炸问题）:
   $$L = \text{MSE}(\hat{IV}, IV) + \lambda_{\text{deriv}} \cdot \text{SmoothL1}\big(\frac{\partial\hat{IV}}{\partial\theta} \times V_{\text{ega}},\; \frac{\partial P}{\partial\theta}\big)$$
   - 网络预测的 IV 梯度 **乘以** Vega 映射回价格空间，与真实价格梯度对比
   - 使用 Smooth L1 (Huber Loss) 替代 MSE，抗离群点
   - 延迟课程学习: 前 20 epoch $\lambda_{\text{deriv}}=0$（纯 IV 拟合）
6. 评估测试集 IV 预测误差
7. 保存模型权重至 `models/heston_ddn_weights.pth`

In [1]:
## Section 1: 导入依赖与项目路径配置
import sys
import os
from pathlib import Path

# 将 modules 目录加入 Python 搜索路径
PROJECT_ROOT = Path(__file__).parent if "__file__" in dir() else Path.cwd()
# 当在 Notebook 中运行时，PROJECT_ROOT 为 Heston_Project/
MODULES_DIR = PROJECT_ROOT / "modules"
sys.path.insert(0, str(PROJECT_ROOT))

from modules.model import HestonDDN
from modules.pricing import generate_training_data, compute_iv_vega_batch, check_feller

import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ---- 路径常量 ----
DATA_PATH  = PROJECT_ROOT.parent / "data" / "heston_dataset_200k.csv"
MODELS_DIR = PROJECT_ROOT / "models"
MODEL_PATH = MODELS_DIR / "heston_ddn_weights.pth"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_PATH    : {DATA_PATH}")
print(f"MODEL_PATH   : {MODEL_PATH}")

PROJECT_ROOT : /Users/liaojiansong/calibration/Heston_Project
DATA_PATH    : /Users/liaojiansong/calibration/Heston_Project/data/heston_dataset_200k.csv
MODEL_PATH   : /Users/liaojiansong/calibration/Heston_Project/models/heston_ddn_weights.pth


In [2]:
## Section 2: 加载或生成训练数据集（乘法 Trick 版本）
#
# 【乘法 Trick 核心思路】
#   旧做法: true_grad_iv = d_xxx * S0 / vega  ← Vega→0 时除以零灾难！
#   新做法: 直接加载 d_xxx（价格空间梯度）和 vega，在 Loss 中做乘法:
#           pred_grad_price = (∂IV_hat/∂θ) × vega  vs  true_grad_price = d_xxx
#           永远不需要除以 vega！

# ── 数据必需的列 ──
REQUIRED_COLS = [
    'kappa', 'lambda', 'sigma', 'rho', 'v0', 'r', 'tau', 'S0',
    'K', 'Price', 'log_K_S0', 'price_norm',
    'iv', 'vega',                                      # ★ IV 目标 + Vega（乘法 Trick 需要）
    'd_kappa', 'd_lambda', 'd_sigma', 'd_rho', 'd_v0', # ★ 价格空间梯度 (∂P/∂θ)/S0
]

FALLBACK_PATH = PROJECT_ROOT.parent / "data" / "heston_dataset_200k.csv"

if DATA_PATH.exists():
    src = DATA_PATH
elif FALLBACK_PATH.exists():
    src = FALLBACK_PATH
    print(f"⚠️  data/ 下无 CSV，使用备用路径: {src}")
else:
    src = None

if src is not None:
    print(f"📂 正在加载: {src}")
    df_all = pd.read_csv(src)
    print(f"✅ 加载完成: {len(df_all):,} 条，列: {list(df_all.columns)}")

    # ── 检查是否缺少 IV 列，如果是旧格式则自动增补 ──
    if 'iv' not in df_all.columns:
        print("\n⚠️  旧格式 CSV（无 IV 列），正在用 py_vollib 增补 IV 和 Vega ...")
        iv_arr, vega_arr = compute_iv_vega_batch(
            df_all['Price'].values,
            df_all['S0'].values,
            df_all['K'].values,
            df_all['tau'].values,
            df_all['r'].values,
        )
        df_all['iv']   = iv_arr
        df_all['vega'] = vega_arr

        # 过滤 IV/Vega 为 NaN 或 ≤ 0 的脏数据
        n_before = len(df_all)
        df_all = df_all.dropna(subset=['iv', 'vega'])
        df_all = df_all[(df_all['iv'] > 0) & (df_all['vega'] > 0)].copy()
        print(f"  清洗: {n_before:,} → {len(df_all):,} (丢弃 {n_before - len(df_all):,} 条 IV/Vega 无效数据)")

        # 过滤不满足 Feller 条件的数据
        n_before = len(df_all)
        df_all['feller_ok'] = df_all.apply(
            lambda row: check_feller(row['kappa'], row['lambda'], row['sigma']), axis=1
        )
        df_all = df_all[df_all['feller_ok']].drop(columns=['feller_ok']).copy()
        print(f"  Feller 过滤: {n_before:,} → {len(df_all):,} (丢弃 {n_before - len(df_all):,} 条)")
        print("  ✅ IV 和 Vega 增补完成")

    # ── 检查价格空间梯度列是否存在 ──
    price_grad_cols = ['d_kappa', 'd_lambda', 'd_sigma', 'd_rho', 'd_v0']
    missing_grads = [c for c in price_grad_cols if c not in df_all.columns]
    if missing_grads:
        raise ValueError(
            f"❌ CSV 缺少价格空间梯度列: {missing_grads}\n"
            f"   乘法 Trick 需要 d_xxx = (∂Price/∂θ)/S0 列。请重新生成训练数据。"
        )

    # ── 清洗价格空间梯度: 去除 inf 和 NaN ──
    df_all[price_grad_cols] = df_all[price_grad_cols].replace([np.inf, -np.inf], np.nan)
    n_before = len(df_all)
    df_all = df_all.dropna(subset=price_grad_cols + ['iv', 'vega']).reset_index(drop=True)
    df_all = df_all[(df_all['iv'] > 0) & (df_all['vega'] > 0)].copy()
    print(f"  梯度+IV+Vega 清洗: {n_before:,} → {len(df_all):,}")

    df_all = df_all.reset_index(drop=True)
    print(f"\n✅ 最终数据集: {len(df_all):,} 条")
else:
    print("⚠️  未找到 CSV，正在生成训练数据（IV 目标版本，需较长时间）...")
    df_all = generate_training_data(num_samples=200_000)
    df_all.to_csv(DATA_PATH, index=False)
    print(f"💾 数据已保存至: {DATA_PATH}")

# 关键列统计
print()
print("IV & Vega 统计:")
print(df_all[['iv', 'vega']].describe().round(4))
print()
print("价格空间梯度 d_xxx 统计（乘法 Trick 使用这些列）:")
print(df_all[['d_kappa', 'd_sigma', 'd_rho']].describe().round(6))
print()
print("💡 乘法 Trick: L_grad = SmoothL1( (∂IV_hat/∂θ)×Vega , d_xxx×S0 )")
print("   → 永远不除以 Vega，深度 OTM/ITM 期权不会导致梯度爆炸")

📂 正在加载: /Users/liaojiansong/calibration/Heston_Project/data/heston_dataset_200k.csv
✅ 加载完成: 199,989 条，列: ['kappa', 'lambda', 'sigma', 'rho', 'v0', 'r', 'tau', 'S0', 'K', 'Price', 'd_kappa', 'd_lambda', 'd_sigma', 'd_rho', 'd_v0', 'log_K_S0', 'price_norm']

⚠️  旧格式 CSV（无 IV 列），正在用 py_vollib 增补 IV 和 Vega ...
✅ 加载完成: 199,989 条，列: ['kappa', 'lambda', 'sigma', 'rho', 'v0', 'r', 'tau', 'S0', 'K', 'Price', 'd_kappa', 'd_lambda', 'd_sigma', 'd_rho', 'd_v0', 'log_K_S0', 'price_norm']

⚠️  旧格式 CSV（无 IV 列），正在用 py_vollib 增补 IV 和 Vega ...


/Users/liaojiansong/anaconda3/lib/python3.13/site-packages/py_vollib_vectorized/implied_volatility.py:75: UserWarning: Found Below Intrinsic contracts at index [413, 500, 950, 1185, 1951, 2113, 2413, 3023, 3249, 3269, 3457, 3637, 4011, 4111, 4299, 5541, 5868, 5892, 6806, 6869, 7132, 7232, 7460, 7899, 8390, 9034, 9537, 9558, 9602, 9727, 10095, 11069, 11192, 12280, 13070, 13080, 13737, 13871, 14596, 15032, 15583, 15770, 15845, 15891, 16218, 16419, 16689, 16813, 16979, 17244, 18265, 18971, 19090, 19114, 19356, 19421, 19613, 20130, 20500, 21279, 21597, 22511, 22609, 23049, 23423, 23505, 23699, 24490, 24795, 25916, 26441, 26986, 26998, 27322, 28179, 28233, 28394, 28467, 28810, 28964, 29013, 29620, 29839, 30281, 30514, 30542, 31045, 31190, 31222, 33021, 33323, 34555, 34744, 35264, 35792, 36218, 36743, 37546, 37696, 38129, 38342, 38406, 38418, 39072, 39608, 40040, 40444, 40671, 40746, 40799, 41079, 41098, 41302, 41703, 41985, 42405, 42457, 42864, 43247, 44261, 44901, 44974, 45803, 46092, 4629

  清洗: 199,989 → 199,426 (丢弃 563 条 IV/Vega 无效数据)
  Feller 过滤: 199,426 → 172,206 (丢弃 27,220 条)
  ✅ IV 和 Vega 增补完成
  梯度+IV+Vega 清洗: 172,206 → 172,206

✅ 最终数据集: 172,206 条

IV & Vega 统计:
                iv         vega
count  172206.0000  172206.0000
mean        0.6903       6.5753
std         0.1634       5.1097
min         0.0872       0.0000
25%         0.5805       2.2783
50%         0.7095       5.5221
75%         0.8178       9.9183
max         1.0386      23.8121

价格空间梯度 d_xxx 统计（乘法 Trick 使用这些列）:
             d_kappa        d_sigma          d_rho
count  172206.000000  172206.000000  172206.000000
mean        0.001913      -0.006250       0.004512
std         0.010889       0.009776       0.009171
min        -0.069454      -0.087711      -0.014069
25%        -0.002330      -0.010584      -0.001646
50%         0.000384      -0.003691       0.001892
75%         0.004694       0.000414       0.008431
max         0.229147       0.023235       0.074734

💡 乘法 Trick: L_grad = SmoothL1( (∂IV_

In [3]:
## Section 3: 数据集划分（70:15:15）与 Tensor 转换
#
# 【乘法 Trick 数据接口变更】
#   旧接口: (X, y_iv, g_iv)          g_iv = div_xxx (IV 空间梯度，Vega→0 时爆炸)
#   新接口: (X, y_iv, vega, g_price)  vega + d_xxx (价格空间梯度，数值稳定)

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
BATCH_SIZE  = 256

device = torch.device(
    "mps"  if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"使用设备: {device}")

# 随机打乱
df_all = df_all.sample(frac=1, random_state=42).reset_index(drop=True)
n_total = len(df_all)
n_train = int(n_total * TRAIN_RATIO)
n_val   = int(n_total * VAL_RATIO)

df_train = df_all.iloc[:n_train]
df_val   = df_all.iloc[n_train : n_train + n_val]
df_test  = df_all.iloc[n_train + n_val :]

print(f"Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}")

# ── 特征列定义（8维: 利用齐次性移除 S0） ──
feat_cols       = ['kappa', 'lambda', 'sigma', 'rho', 'v0', 'r', 'tau', 'log_K_S0']
price_grad_cols = ['d_kappa', 'd_lambda', 'd_sigma', 'd_rho', 'd_v0']

def df_to_tensors(df, dev):
    """
    将 DataFrame 转换为 (X, y_iv, vega, g_price) 四元组 Tensor。

    返回:
      X       : (N, 8)  8维输入特征
      y_iv    : (N, 1)  真实 BS 隐含波动率（训练目标）
      vega    : (N, 1)  真实 BS Vega（乘法 Trick 中映射 IV→Price 空间）
      g_price : (N, 5)  价格空间梯度 d_xxx = (∂Price/∂θ_i) / S0
    """
    X       = torch.tensor(df[feat_cols].values,       dtype=torch.float32).to(dev)
    y_iv    = torch.tensor(df['iv'].values,            dtype=torch.float32).view(-1, 1).to(dev)
    vega    = torch.tensor(df['vega'].values,          dtype=torch.float32).view(-1, 1).to(dev)
    g_price = torch.tensor(df[price_grad_cols].values, dtype=torch.float32).to(dev)
    return X, y_iv, vega, g_price

X_train, y_train, vega_train, g_price_train = df_to_tensors(df_train, device)
X_val,   y_val,   vega_val,   g_price_val   = df_to_tensors(df_val,   device)
X_test,  y_test,  vega_test,  g_price_test  = df_to_tensors(df_test,  device)

print(f"X_train shape: {X_train.shape}")
print(f"y_train (IV) shape: {y_train.shape}, range: [{y_train.min():.4f}, {y_train.max():.4f}]")
print(f"vega_train shape: {vega_train.shape}, range: [{vega_train.min():.4f}, {vega_train.max():.4f}]")
print(f"g_price_train (d_xxx) shape: {g_price_train.shape}")
print()
print("💡 数据接口: (X, y_iv, vega, g_price) — 乘法 Trick 四元组")

使用设备: mps
Train: 120,544  Val: 25,830  Test: 25,832
X_train shape: torch.Size([120544, 8])
y_train (IV) shape: torch.Size([120544, 1]), range: [0.0932, 1.0386]
vega_train shape: torch.Size([120544, 1]), range: [0.0000, 23.7813]
g_price_train (d_xxx) shape: torch.Size([120544, 5])

💡 数据接口: (X, y_iv, vega, g_price) — 乘法 Trick 四元组


In [4]:
## Section 4: 模型初始化与归一化 Scaler 拟合

EPOCHS        = 200
LEARNING_RATE = 0.001
DECAY_GAMMA   = 0.9
DECAY_STEP    = 20        # 每 20 epoch 学习率 × 0.9

# ── 延迟梯度训练策略（课程学习 Warm-up） ──
# 前 WARMUP_EPOCHS 个 epoch: λ_deriv = 0（纯 IV 拟合，建立稳定基线）
# 之后: λ_deriv = LAMBDA_DERIV_FINAL（加入乘法 Trick 梯度平滑）
WARMUP_EPOCHS      = 20
LAMBDA_DERIV_FINAL = 1

model = HestonDDN(input_dim=8).to(device)
model.fit_scalers(X_train, y_train)   # y_train 现在是 IV，冻结归一化极值

# ── DataLoader: 四元组 (X, y_iv, vega, g_price) ──
train_dataset = TensorDataset(X_train, y_train, vega_train, g_price_train)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=DECAY_STEP, gamma=DECAY_GAMMA)

print(f"模型参数量: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"批次数/epoch: {len(train_loader)}")
print(f"归一化极值 y_min(IV)={model.y_min.item():.6f}, y_max(IV)={model.y_max.item():.6f}")
print(f"延迟梯度策略: 前 {WARMUP_EPOCHS} epoch λ_deriv=0, 之后 λ_deriv={LAMBDA_DERIV_FINAL}")
print(f"梯度 Loss 类型: Smooth L1 (Huber Loss) — 对离群点鲁棒")
print(f"梯度空间: 价格空间（乘法 Trick: pred_grad_iv × vega vs true_grad_price）")

模型参数量: 114,751
批次数/epoch: 471
归一化极值 y_min(IV)=0.093227, y_max(IV)=1.038616
延迟梯度策略: 前 20 epoch λ_deriv=0, 之后 λ_deriv=1
梯度 Loss 类型: Smooth L1 (Huber Loss) — 对离群点鲁棒
梯度空间: 价格空间（乘法 Trick: pred_grad_iv × vega vs true_grad_price）


In [5]:
## Section 5: 训练循环 — IV 目标 + 乘法 Trick 梯度 Loss + 延迟课程学习

t0 = time.time()
print(f"{'Epoch':>6s} | {'λ_deriv':>8s} | {'Train Loss':>12s} | {'L_IV':>10s} | {'L_Grad':>10s} | "
      f"{'Val IV Err%':>12s} | {'LR':>10s} | {'Elapsed':>8s}")
print("-" * 95)

for epoch in range(1, EPOCHS + 1):
    # ── 延迟梯度训练: 前 WARMUP_EPOCHS 纯 IV 拟合，之后加入乘法 Trick 梯度平滑 ──
    if epoch <= WARMUP_EPOCHS:
        lambda_deriv = 0.0
    else:
        lambda_deriv = LAMBDA_DERIV_FINAL

    model.train()
    epoch_loss = 0.0
    for b_X, b_y, b_vega, b_gp in train_loader:
        # b_X:    (batch, 8)  输入特征
        # b_y:    (batch, 1)  真实 IV
        # b_vega: (batch, 1)  真实 BS Vega（乘法 Trick 的桥梁）
        # b_gp:   (batch, 5)  真实价格空间梯度 d_xxx
        optimizer.zero_grad()
        loss, l_iv, l_g = model.compute_loss(
            b_X, b_y, b_vega, b_gp, lambda_deriv=lambda_deriv
        )
        loss.backward()
        # 梯度裁剪: 防止 autograd 二阶导数偶尔过大
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    avg_loss = epoch_loss / len(train_loader)

    if epoch % 20 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            # 验证集: DDN 预测 IV vs 真实 IV
            pred_iv_val = model(X_val)                    # (N, 1) 预测 IV
            rel_err = (torch.abs(pred_iv_val - y_val) /
                       (torch.abs(y_val) + 1e-6) * 100)
            mean_rel = rel_err.mean().item()
        elapsed = time.time() - t0
        lr_now  = scheduler.get_last_lr()[0]
        print(f"{epoch:6d} | {lambda_deriv:8.4f} | {avg_loss:12.6f} | {l_iv.item():10.6f} | {l_g.item():10.6f} | "
              f"{mean_rel:12.4f} | {lr_now:10.6f} | {elapsed:7.1f}s")

print(f"\n⏱ 训练完成，总耗时: {time.time()-t0:.1f}s")

 Epoch |  λ_deriv |   Train Loss |       L_IV |     L_Grad |  Val IV Err% |         LR |  Elapsed
-----------------------------------------------------------------------------------------------
     1 |   0.0000 |     0.022871 |   0.004703 |   0.000000 |       8.9544 |   0.001000 |     3.8s
     1 |   0.0000 |     0.022871 |   0.004703 |   0.000000 |       8.9544 |   0.001000 |     3.8s
    20 |   0.0000 |     0.000490 |   0.000383 |   0.000000 |       2.1394 |   0.000900 |    51.1s
    20 |   0.0000 |     0.000490 |   0.000383 |   0.000000 |       2.1394 |   0.000900 |    51.1s
    40 |   1.0000 |     0.001072 |   0.000047 |   0.001251 |       0.9401 |   0.000810 |   102.6s
    40 |   1.0000 |     0.001072 |   0.000047 |   0.001251 |       0.9401 |   0.000810 |   102.6s
    60 |   1.0000 |     0.001030 |   0.000056 |   0.001040 |       0.8741 |   0.000729 |   152.5s
    60 |   1.0000 |     0.001030 |   0.000056 |   0.001040 |       0.8741 |   0.000729 |   152.5s
    80 |   1.0000 |   

In [6]:
## Section 6: 测试集评估 — IV 预测误差统计

model.eval()
with torch.no_grad():
    pred_iv_test = model(X_test)          # (N, 1) 预测 IV

    abs_err = torch.abs(pred_iv_test - y_test)
    rel_err = abs_err / (torch.abs(y_test) + 1e-6) * 100

# 前 15 条样本明细
print(f"  {'#':>5s} | {'真实 IV':>10s} | {'预测 IV':>10s} | {'绝对误差':>10s} | {'相对误差':>8s}")
print("  " + "-" * 55)
for i in range(min(15, len(y_test))):
    ti = y_test[i].item()
    pi = pred_iv_test[i].item()
    ae = abs_err[i].item()
    re = rel_err[i].item()
    flag = "✅" if re < 5.0 else "❌"
    print(f"  {flag}{i+1:4d} | {ti:10.6f} | {pi:10.6f} | {ae:10.6f} | {re:7.2f}%")

# 全量统计
print()
print(f"  📊 Test set ({len(y_test):,} 条) IV 预测统计:")
print(f"     均值相对误差   : {rel_err.mean().item():.4f}%")
print(f"     中位数相对误差 : {rel_err.median().item():.4f}%")
print(f"     最大相对误差   : {rel_err.max().item():.4f}%")
print(f"     相对误差 < 1%  : {(rel_err < 1.0).float().mean().item()*100:.1f}%")
print(f"     相对误差 < 5%  : {(rel_err < 5.0).float().mean().item()*100:.1f}%")
print(f"     MAE (IV)       : {abs_err.mean().item():.6f}")
print(f"     RMSE (IV)      : {(abs_err**2).mean().sqrt().item():.6f}")

      # |      真实 IV |      预测 IV |       绝对误差 |     相对误差
  -------------------------------------------------------
  ✅   1 |   0.791255 |   0.790879 |   0.000376 |    0.05%
  ✅   2 |   0.278354 |   0.268810 |   0.009544 |    3.43%
  ✅   3 |   0.834724 |   0.853420 |   0.018696 |    2.24%
  ✅   4 |   0.861310 |   0.863627 |   0.002317 |    0.27%
  ✅   5 |   0.968995 |   0.974171 |   0.005175 |    0.53%
  ✅   6 |   0.589025 |   0.591413 |   0.002388 |    0.41%
  ✅   7 |   0.655816 |   0.655806 |   0.000010 |    0.00%
  ✅   8 |   0.882527 |   0.881798 |   0.000729 |    0.08%
  ✅   9 |   0.815463 |   0.817000 |   0.001537 |    0.19%
  ✅  10 |   0.738521 |   0.738179 |   0.000342 |    0.05%
  ✅  11 |   0.803642 |   0.801194 |   0.002448 |    0.30%
  ✅  12 |   0.799492 |   0.800414 |   0.000923 |    0.12%
  ✅  13 |   0.747476 |   0.744370 |   0.003105 |    0.42%
  ✅  14 |   0.887758 |   0.885358 |   0.002401 |    0.27%
  ✅  15 |   0.742256 |   0.743898 |   0.001642 |    0.22%

  📊 Test set 

In [7]:
## Section 7: 保存模型权重

torch.save({
    'model_state_dict': model.state_dict(),   # 包含 register_buffer 的归一化极值
    'is_fitted':  model.is_fitted,
    'input_dim':  model.input_dim,
    'heston_dim': model.heston_dim,
    'target':     'iv',                        # ★ 标记目标为 IV（与旧版 price_norm 区分）
}, MODEL_PATH)

size_kb = MODEL_PATH.stat().st_size / 1024
print(f"💾 模型已保存: {MODEL_PATH}  ({size_kb:.1f} KB)")
print(f"   目标: IV (隐含波动率)")
print(f"   归一化极值:")
print(f"     X_min = {model.X_min.cpu().numpy()}")
print(f"     X_max = {model.X_max.cpu().numpy()}")
print(f"     y_min (IV) = {model.y_min.item():.6f}")
print(f"     y_max (IV) = {model.y_max.item():.6f}")
print()
print("✅ 训练全流程完成！（IV 目标 + Feller 条件预过滤）")

💾 模型已保存: /Users/liaojiansong/calibration/Heston_Project/models/heston_ddn_weights.pth  (454.7 KB)
   目标: IV (隐含波动率)
   归一化极值:
     X_min = [ 0.01369767  0.01011868  0.10000127 -0.8999977   0.01000155  0.00100037
  0.05000845 -0.49998844]
     X_max = [ 4.9999504   0.99999684  0.99998355 -0.05000427  0.99999714  0.09999973
  0.9999982   0.49997777]
     y_min (IV) = 0.093227
     y_max (IV) = 1.038616

✅ 训练全流程完成！（IV 目标 + Feller 条件预过滤）
